In [1]:
# Research review: Recovered exploratory extraction workflow. Cells share notebook state; run order and array-axis conventions require review before rerunning.
import os
import re
import h5py
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

In [2]:
ROOT_FOLDER = "RO_data"

In [9]:
import os

ROOT_FOLDER = "RO_data"

count = 0

for root, dirs, files in os.walk(ROOT_FOLDER):

    for file in files:

        if file.endswith(".h5"):

            count += 1

            print(os.path.join(root,file))

print("Total H5 files:",count)

RO_data\configuration1_interspacer0.75mm\HR_spacer_images\100_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\101_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\102_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\103_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\104_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\105_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\106_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\107_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\108_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\109_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\10_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\110_spacer_image.h5
RO_data\configuration1_interspacer0.75mm\HR_spacer_images\111_spa

In [3]:
def read_xmf_mapping(xmf_file):

    tree = ET.parse(xmf_file)
    root = tree.getroot()

    mapping = {}

    for attribute in root.iter():

        if "Attribute" in attribute.tag:

            variable = attribute.attrib.get("Name")

            for child in attribute:

                if "DataItem" in child.tag:

                    text = child.text.strip()

                    dataset = text.split(":")[-1].replace("/", "")

                    mapping[variable] = dataset

    return mapping

In [4]:
master = []

In [5]:
# Research review: The arrays retained after this loop correspond to its last case. Later cells are outside this loop; do not interpret their row-level export as covering every case.
for root, dirs, files in os.walk(ROOT_FOLDER):

    h5_files = [f for f in files if f.endswith(".h5")]

    for h5_file in h5_files:

        base = os.path.splitext(h5_file)[0]

        xmf_file = base + ".xmf"

        xmf_path = os.path.join(root, xmf_file)

        h5_path = os.path.join(root, h5_file)

        if not os.path.exists(xmf_path):

            print("Missing XMF:", xmf_file)

            continue

        print("Reading:", h5_file)

        # Configuration

        config = int(re.search(r'configuration(\d+)', root).group(1))

        # Resolution

        resolution = "HR" if "HR" in root.upper() else "LR"

        # Image Number

        image = int(re.findall(r'\d+', h5_file)[0])

        mapping = read_xmf_mapping(xmf_path)

        with h5py.File(h5_path, 'r') as h5:

            C = h5[mapping["C"]][:]

            U = h5[mapping["U"]][:]

            P = h5[mapping["p"]][:]

            PRGH = h5[mapping["p_rgh"]][:]

            RHO = h5[mapping["rho"]][:]

            VISC = h5[mapping["visc"]][:]

Reading: 100_spacer_image.h5
Reading: 101_spacer_image.h5
Reading: 102_spacer_image.h5
Reading: 103_spacer_image.h5
Reading: 104_spacer_image.h5
Reading: 105_spacer_image.h5
Reading: 106_spacer_image.h5
Reading: 107_spacer_image.h5
Reading: 108_spacer_image.h5
Reading: 109_spacer_image.h5
Reading: 10_spacer_image.h5
Reading: 110_spacer_image.h5
Reading: 111_spacer_image.h5
Reading: 112_spacer_image.h5
Reading: 113_spacer_image.h5
Reading: 114_spacer_image.h5
Reading: 115_spacer_image.h5
Reading: 116_spacer_image.h5
Reading: 117_spacer_image.h5
Reading: 118_spacer_image.h5
Reading: 119_spacer_image.h5
Reading: 11_spacer_image.h5
Reading: 120_spacer_image.h5
Reading: 121_spacer_image.h5
Reading: 122_spacer_image.h5
Reading: 123_spacer_image.h5
Reading: 124_spacer_image.h5
Reading: 125_spacer_image.h5
Reading: 126_spacer_image.h5
Reading: 127_spacer_image.h5
Reading: 128_spacer_image.h5
Reading: 129_spacer_image.h5
Reading: 12_spacer_image.h5
Reading: 130_spacer_image.h5
Reading: 131_spac

In [6]:
            Ux = U[:,:,:,0]

            Uy = U[:,:,:,1]

            Uz = U[:,:,:,2]

            Velocity = np.sqrt(Ux**2 + Uy**2 + Uz**2)

In [7]:
            nx = C.shape[0]

            ny = C.shape[2]

            for i in range(nx):

                for j in range(ny):

                    row = {}

                    row["Configuration"] = config
                    row["Resolution"] = resolution
                    row["Image"] = image

                    row["X"] = i
                    row["Y"] = j

                    row["Concentration"] = C[i,0,j]

                    row["Ux"] = Ux[i,0,j]
                    row["Uy"] = Uy[i,0,j]
                    row["Uz"] = Uz[i,0,j]

                    row["Velocity"] = Velocity[i,0,j]

                    row["Pressure"] = P[i,0,j]

                    row["Pressure_rgh"] = PRGH[i,0,j]

                    row["Density"] = RHO[i,0,j]

                    row["Viscosity"] = VISC[i,0,j]

                    master.append(row)

In [8]:
df = pd.DataFrame(master)

df.sort_values(
    by=[
        "Configuration",
        "Resolution",
        "Image",
        "X",
        "Y"
    ],
    inplace=True
)

df.to_excel("Raw_CFD_Data.xlsx", index=False)

print(df.head())

print()

print(df.shape)

   Configuration Resolution  Image  X  Y  Concentration        Ux  \
0              6         LR      9  0  0     118.224335  0.000001   
1              6         LR      9  0  1     115.978447  0.000001   
2              6         LR      9  0  2     114.873909  0.000001   
3              6         LR      9  0  3     114.411125  0.000002   
4              6         LR      9  0  4     113.967552  0.000002   

             Uy        Uz  Velocity    Pressure  Pressure_rgh      Density  \
0  3.703757e-25 -0.000015  0.000016  20003700.0    20003700.0  1067.505005   
1  3.703757e-25 -0.000016  0.000016  20003700.0    20003700.0  1068.083496   
2 -1.181219e-26 -0.000016  0.000016  20003700.0    20003700.0  1068.311279   
3  3.108278e-25 -0.000016  0.000016  20003700.0    20003700.0  1068.334961   
4  3.108278e-25 -0.000016  0.000016  20003700.0    20003700.0  1068.334961   

   Viscosity  
0   0.001186  
1   0.001188  
2   0.001188  
3   0.001189  
4   0.001189  

(400, 14)
